# Embedding Analysis for Fraud Detection

This notebook generates and analyzes transaction embeddings for the Sparkov fraud
detection dataset. Embeddings are dense vector representations that capture semantic
similarity between transactions -- transactions with similar characteristics should
cluster together in embedding space.

**Goals:**

1. Generate real embeddings from transaction text using the local sentence-transformer
   model (all-MiniLM-L6-v2, 384 dimensions). Falls back to random vectors if
   sentence-transformers is not installed.
2. Visualize the embedding space in 2D (t-SNE) and 3D (PCA) to assess
   fraud vs. legitimate separation.
3. Analyze PCA explained variance to understand embedding dimensionality.
4. Compute cosine distance distributions between fraud and legitimate clusters.
5. Evaluate embedding quality using silhouette score and nearest-neighbor purity.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist

from src.data.loader import SparkovDataLoader
from src.visualization.plots import (
    plot_embedding_space_2d,
    plot_embedding_space_3d,
    plot_pca_explained_variance,
    plot_cosine_distance_distribution,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Libraries loaded successfully.")

In [ ]:
# Load the Sparkov dataset
loader = SparkovDataLoader(data_path="../data/fraudTrain.csv")
df = loader.load()

# Subsample for computational tractability in this demo
np.random.seed(42)
n_samples = 5000
fraud_df = df[df["is_fraud"] == 1]
legit_df = df[df["is_fraud"] == 0]

# Stratified sample: keep fraud proportion but ensure enough fraud examples
n_fraud = min(len(fraud_df), 500)
n_legit = n_samples - n_fraud

sample_fraud = fraud_df.sample(n=n_fraud, random_state=42)
sample_legit = legit_df.sample(n=n_legit, random_state=42)
df_sample = pd.concat([sample_fraud, sample_legit]).reset_index(drop=True)

labels = df_sample["is_fraud"].values
print(f"Sample size: {len(df_sample)} (fraud: {labels.sum()}, legit: {(1 - labels).sum():.0f})")

In [ ]:
# Generate transaction text for each sample row using the loader utility
texts = loader.generate_transaction_texts(df_sample).tolist()
print(f"Generated {len(texts)} transaction text strings.")
print(f"Example: {texts[0][:120]}...")

In [ ]:
# Generate embeddings using the local sentence-transformer model (all-MiniLM-L6-v2, 384-d).
# Falls back to random vectors if sentence-transformers is not installed.

try:
    from src.embeddings.generator import LocalEmbeddingGenerator

    generator = LocalEmbeddingGenerator()
    embeddings = generator.encode_texts(texts)
    EMBEDDING_DIM = embeddings.shape[1]
    print(f"Using LocalEmbeddingGenerator (model: all-MiniLM-L6-v2, dim={EMBEDDING_DIM})")
except ImportError:
    print("WARNING: sentence-transformers is not installed. Falling back to random embeddings.")
    EMBEDDING_DIM = 384
    rng = np.random.default_rng(seed=42)
    embeddings = rng.standard_normal((len(df_sample), EMBEDDING_DIM)).astype(np.float32)

# Normalize to unit length (common for cosine-similarity-based retrieval)
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / norms

print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Norm of first embedding: {np.linalg.norm(embeddings[0]):.4f}")

## Embedding Space Visualization

Dimensionality reduction allows us to visually assess whether fraud and legitimate
transactions form distinct clusters in embedding space. Good embeddings should show
at least partial separation between the two classes.

In [ ]:
# 2D visualization using t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(embeddings)

fig = plot_embedding_space_2d(
    embeddings_2d=embeddings_2d,
    labels=labels,
    title="t-SNE Projection of Transaction Embeddings",
    label_names={0: "Legitimate", 1: "Fraud"},
)
plt.tight_layout()
plt.show()

In [ ]:
# 3D visualization using PCA
pca_3d = PCA(n_components=3, random_state=42)
embeddings_3d = pca_3d.fit_transform(embeddings)

fig = plot_embedding_space_3d(
    embeddings_3d=embeddings_3d,
    labels=labels,
    title="PCA 3D Projection of Transaction Embeddings",
    label_names={0: "Legitimate", 1: "Fraud"},
)
plt.tight_layout()
plt.show()

print(f"Variance explained by first 3 PCs: {pca_3d.explained_variance_ratio_.sum():.2%}")

## PCA Analysis

Examining the explained variance across principal components tells us about the
effective dimensionality of the embedding space. If a small number of components
capture most variance, the embeddings live on a lower-dimensional manifold,
which simplifies downstream drift detection.

In [ ]:
# Full PCA to see explained variance curve
pca_full = PCA(n_components=min(50, EMBEDDING_DIM), random_state=42)
pca_full.fit(embeddings)

fig = plot_pca_explained_variance(
    explained_variance_ratio=pca_full.explained_variance_ratio_,
    title="PCA Explained Variance Across Components",
)
plt.tight_layout()
plt.show()

cumulative = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = np.searchsorted(cumulative, 0.90) + 1
n_95 = np.searchsorted(cumulative, 0.95) + 1
print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")

## Cosine Distance Analysis

Cosine distance measures the angular separation between embedding vectors.
For fraud detection, we expect:

- **Within-class distances** (fraud-fraud, legit-legit) to be relatively small.
- **Between-class distances** (fraud-legit) to be larger.

The ratio of between-class to within-class distance indicates how well the
embedding space separates fraud from legitimate transactions.

In [ ]:
# Compute cosine distances
fraud_mask = labels == 1
fraud_embeddings = embeddings[fraud_mask]
legit_embeddings = embeddings[~fraud_mask]

# Within-class: sample pairwise distances (full computation is O(n^2))
n_pairs = 2000

def sample_pairwise_cosine(emb, n_pairs, rng):
    """Sample pairwise cosine distances from an embedding matrix."""
    n = emb.shape[0]
    idx_a = rng.integers(0, n, size=n_pairs)
    idx_b = rng.integers(0, n, size=n_pairs)
    # Avoid self-pairs
    mask = idx_a != idx_b
    idx_a, idx_b = idx_a[mask], idx_b[mask]
    dists = 1.0 - np.sum(emb[idx_a] * emb[idx_b], axis=1)
    return dists

rng_dist = np.random.default_rng(seed=123)
fraud_fraud_dists = sample_pairwise_cosine(fraud_embeddings, n_pairs, rng_dist)
legit_legit_dists = sample_pairwise_cosine(legit_embeddings, n_pairs, rng_dist)

# Between-class distances
idx_f = rng_dist.integers(0, fraud_embeddings.shape[0], size=n_pairs)
idx_l = rng_dist.integers(0, legit_embeddings.shape[0], size=n_pairs)
fraud_legit_dists = 1.0 - np.sum(fraud_embeddings[idx_f] * legit_embeddings[idx_l], axis=1)

fig = plot_cosine_distance_distribution(
    within_fraud=fraud_fraud_dists,
    within_legit=legit_legit_dists,
    between_classes=fraud_legit_dists,
    title="Cosine Distance Distributions",
)
plt.tight_layout()
plt.show()

print(f"Mean within-fraud distance:   {fraud_fraud_dists.mean():.4f}")
print(f"Mean within-legit distance:   {legit_legit_dists.mean():.4f}")
print(f"Mean between-class distance:  {fraud_legit_dists.mean():.4f}")
print(f"Separation ratio (between / avg within): "
      f"{fraud_legit_dists.mean() / ((fraud_fraud_dists.mean() + legit_legit_dists.mean()) / 2):.2f}")

## Embedding Quality Assessment

We use two quantitative metrics to assess whether the embedding space is suitable
for fraud detection:

- **Silhouette Score:** Ranges from -1 to 1. Values above 0.1 indicate that
  fraud and legitimate clusters have meaningful separation.
- **Nearest-Neighbor Purity:** Fraction of each point's k nearest neighbors
  that share the same label. Values above 0.5 (random baseline for binary
  classification) indicate that embeddings encode class-relevant structure.

In [ ]:
# Silhouette score
# Use a subsample if the dataset is large, to keep computation feasible
sil_sample_size = min(3000, len(embeddings))
sil_idx = rng_dist.choice(len(embeddings), size=sil_sample_size, replace=False)
sil_embeddings = embeddings[sil_idx]
sil_labels = labels[sil_idx]

sil_score = silhouette_score(sil_embeddings, sil_labels, metric="cosine")
print(f"Silhouette Score (cosine): {sil_score:.4f}")

# Nearest-neighbor purity (k=10)
k = 10
nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine")
nn.fit(sil_embeddings)
distances, indices = nn.kneighbors(sil_embeddings)

# Exclude self (index 0 in the neighbor list)
neighbor_labels = sil_labels[indices[:, 1:]]
purity = (neighbor_labels == sil_labels[:, np.newaxis]).mean()

print(f"Nearest-Neighbor Purity (k={k}): {purity:.4f}")
print()

# Breakdown by class
fraud_idx = sil_labels == 1
legit_idx = sil_labels == 0
fraud_purity = (neighbor_labels[fraud_idx] == 1).mean()
legit_purity = (neighbor_labels[legit_idx] == 0).mean()
print(f"Fraud cluster purity: {fraud_purity:.4f}")
print(f"Legitimate cluster purity: {legit_purity:.4f}")

## Key Takeaways

**What good embeddings look like:**

- Clear visual separation between fraud and legitimate clusters in t-SNE/PCA plots.
- High silhouette score (above 0.1 for a highly imbalanced binary task).
- Nearest-neighbor purity well above the random baseline (0.5 for binary labels).
- Between-class cosine distance substantially larger than within-class distance.
- Compact variance structure: most information captured in the first 10-20 PCA components.

**What bad embeddings look like:**

- Fraud and legitimate points are uniformly mixed in the reduced-dimension projections.
- Silhouette score near zero or negative, indicating no meaningful cluster structure.
- Nearest-neighbor purity close to the class prior (i.e., no better than random).
- Cosine distance distributions for within-class and between-class overlap almost entirely.
- Flat explained variance curve: many components contribute equally, suggesting noise dominates.

**Implications for drift detection:**

When embedding quality is high, we have a well-structured space to monitor.
Drift in this space -- measured by shifts in centroids, changes in cluster geometry,
or distributional divergence -- provides a meaningful signal that the underlying
data distribution has changed. The next notebook will set up and run drift detection
on time-windowed slices of these embeddings.